# coherence-seg Colab 執行入口（規格書 §9.2）
所有 cell 冪等可重跑。依序執行；斷線後重跑 1–6 再用「續訓」cell。

In [ ]:
# 1. GPU 檢查
!nvidia-smi

In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. clone / pull repo + 安裝套件
import os
REPO = 'https://github.com/<YOUR_ACCOUNT>/coherence-seg.git'  # TODO: 換成你的 repo
if not os.path.exists('/content/coherence-seg'):
    !git clone $REPO /content/coherence-seg
%cd /content/coherence-seg
!git pull
!pip install -q -r requirements.txt

In [ ]:
# 4. 從 Colab Secrets 讀 WANDB_API_KEY（金鑰不得寫死在任何檔案）
import os
from google.colab import userdata
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('wandb key 已設定')
except Exception:
    print('未設定 WANDB_API_KEY secret，將以無 logger 模式訓練')

In [ ]:
# 5. 資料同步：Drive → /content 本地（Drive 直讀太慢，§12 陷阱 12）
!mkdir -p /content/data
!cp -rn /content/drive/MyDrive/coherence-seg/data/* /content/data/ || echo '請先把資料放到 Drive: MyDrive/coherence-seg/data/'
!ls /content/data

In [ ]:
# 6. 單元測試
!python -m pytest tests/ -v

In [ ]:
# 7. 訓練（改 config 與 seed）
CONFIG = 'configs/m0_baseline.yaml'
SEED = 42
!python -m src.train --config $CONFIG --seed $SEED

In [ ]:
# 8. 續訓（斷線後使用，自動找 last.ckpt；§9.3）
!python -m src.train --config $CONFIG --seed $SEED --resume

In [ ]:
# 9. 多 seed 批次（已完成 seed 自動跳過）
!bash scripts/run_all_seeds.sh $CONFIG

In [ ]:
# 10. 結果彙整回存 Drive
!python -m src.eval.aggregate --pattern 'results/*.json' --out results/ABLATION.md
!mkdir -p /content/drive/MyDrive/coherence-seg/results
!cp -r results/* /content/drive/MyDrive/coherence-seg/results/